<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Project: Analysis of vacancies from HeadHunter

In [1]:
import pandas as pd
import psycopg2
import warnings

warnings.filterwarnings('ignore')

In [55]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

# Preliminary data analysis

1. Write a query that will count the number of vacancies in our database (vacancies are located in the vacancies table).

In [5]:
# query text
query = f'''
    SELECT *
    FROM public.vacancies
'''

In [6]:
# query result
df = pd.read_sql_query(query, connection)
print('Vacancies in the database:', len(df))
df.head()

Vacancies in the database: 49197


,id,name,key_skills,schedule,experience,employment,salary_from,salary_to,area_id,employer_id
0,55312386,Компьютерный Мастер,Пользователь ПК\tРабота в команде\tРемонт ноут...,Полный день,Нет опыта,Полная занятость,64000.0,NaN,1516,5724811
1,55843014,Системный администратор,Средства криптографической защиты информации\t...,Полный день,От 3 до 6 лет,Полная занятость,NaN,NaN,130,4903713
2,54525964,Lead Java Developer to Poland,Spring Framework\tSQL\tHibernate ORM\tJava\tGit,Удаленная работа,От 3 до 6 лет,Полная занятость,NaN,NaN,160,69961
3,54525965,Lead Java Developer to Poland,Spring Framework\tSQL\tHibernate ORM\tJava\tGit,Удаленная работа,От 3 до 6 лет,Полная занятость,NaN,NaN,159,69961
4,55354053,Специалист службы поддержки с техническими зна...,None,Удаленная работа,Нет опыта,Частичная занятость,15000.0,NaN,1955,1740


2. Write a query that will count the number of employers (employers table).

In [11]:
# query text
query = f'''
    SELECT *
    FROM public.employers
'''

In [12]:
# query result
df = pd.read_sql_query(query, connection)
print('Employers:', len(df))
df.head()

Employers: 23501


,id,name,area
0,2393,"Программный Продукт, ИТ-компания",1
1,72977,БАРС Груп,88
2,3155,"Бест, Торгово-производственная компания, Екате...",3
3,675,КОРУС Консалтинг,2
4,1840010,филиал ФКУ Налог-Сервис ФНС России в Республик...,88


3. Calculate the number of regions using a query (areas table).

In [13]:
# query text
query = f'''
    SELECT *
    FROM public.areas
'''

In [14]:
# query result
df = pd.read_sql_query(query, connection)
print('Regions:', len(df))
df.head()

Regions: 1362


,id,name
0,2758,Тбилиси
1,8,Майкоп
2,1180,Нерюнгри
3,1240,Новокузнецк
4,2,Санкт-Петербург


4. Calculate the number of fields of activity in the database (industries table) using a query.

In [15]:
# query text
query = f'''
    SELECT *
    FROM public.industries
'''

In [16]:
# query result
df = pd.read_sql_query(query, connection)
print('Areas of activity:', len(df))
df.head()

Areas of activity: 294


,id,name
0,7.540,Разработка программного обеспечения
1,7.539,"Системная интеграция, автоматизации технологи..."
2,27.550,Безалкогольные напитки (производство)
3,27.551,"Безалкогольные напитки (продвижение, оптовая т..."
4,13.664,Управление и эксплуатация недвижимости


***

In [69]:
# conclusions from preliminary data analysis

# Unit 4. Detailed analysis of vacancies

1. Write a query to find out the number of job openings (cnt) in each region (area).
Sort by the number of openings in descending order.

In [ ]:
# query text
query = f'''
    SELECT
        a.name AS area,
        COUNT(v.id) AS cnt
    FROM
        public.vacancies v LEFT JOIN
        public.areas a ON v.area_id = a.id
    GROUP BY area
    ORDER BY cnt DESC
'''

In [33]:
# query result
print('Top five by number of vacancies:')
df = pd.read_sql_query(query, connection)
df.head()

Top five by number of vacancies:


,area,cnt
0,Москва,5333
1,Санкт-Петербург,2851
2,Минск,2112
3,Новосибирск,2006
4,Алматы,1892


2. Write a query to determine how many vacancies have at least one of the two salary fields filled.

In [ ]:
# query text
query = f'''
    SELECT COUNT(id)
    FROM public.vacancies
    WHERE
        salary_from IS NOT NULL OR
        salary_to IS NOT NULL
'''

In [45]:
# query result
df = pd.read_sql_query(query, connection)
print('Vacancies have at least one of the two salary fields filled in:', df['count'][0])

Vacancies have at least one of the two salary fields filled in: 24073


3. Find the average values ​​for the lower and upper limits of the salary range. Round the values ​​to the nearest whole number.

In [49]:
# query text
query = f'''
    SELECT
        ROUND(AVG(salary_from), 0) AS lower_bound,
        ROUND(AVG(salary_to), 0) AS upper_bound
    FROM public.vacancies
'''

In [50]:
# query result
df = pd.read_sql_query(query, connection)
df.astype(int)

,lower_bound,upper_bound
0,71065,110537


4. Write a query that will return the number of job postings for each combination of schedule and employment type used in the postings. Sort the results by descending number.

In [54]:
# query text
query = f'''
    SELECT
        schedule,
        employment,
        COUNT(id)
    FROM public.vacancies
    GROUP BY
        schedule,
        employment
    ORDER BY COUNT(id) DESC
'''

In [57]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,schedule,employment,count
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940


5. Write a query that displays the values ​​of the Required Work Experience field in ascending order of the number of vacancies in which this type of experience is indicated.

In [60]:
# query text
query = f'''
    SELECT
        experience,
        COUNT(id)
    FROM public.vacancies
    GROUP BY experience
    ORDER BY COUNT(id)
'''

In [61]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,experience,count
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

In [17]:
# conclusions from a detailed analysis of vacancies

# Unit 5. Employer Analysis

1. Write a query that will allow you to find out which employers are in first and fifth place by the number of vacancies.

In [18]:
# query text
query = f'''
    SELECT
        e.name,
        COUNT(v.id)
    FROM
        public.vacancies v LEFT JOIN
        public.employers e ON v.employer_id = e.id
    GROUP BY e.id
    ORDER BY COUNT(v.id) DESC
'''

In [19]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,name,count
0,Яндекс,1933
1,Ростелеком,491
2,Тинькофф,444
3,СБЕР,428
4,Газпром нефть,331


2. Write a query that will return the number of employers and job openings in each region.
Among the regions with no job openings, find the one with the largest number of employers.

In [34]:
# query text
query = f'''
    SELECT
        a.name,
        COUNT(v.id) AS vacancies,
        COUNT(e.id) AS employers
    FROM
        public.areas a LEFT JOIN
        public.vacancies v ON a.id = v.area_id LEFT JOIN
        public.employers e ON a.id = e.area
    WHERE v.id IS NULL
    GROUP BY a.id
    ORDER BY employers DESC
'''

In [36]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,name,vacancies,employers
0,Россия,0,410
1,Казахстан,0,207
2,Московская область,0,75
3,Краснодарский край,0,19
4,Ростовская область,0,18


3. For each employer, count the number of regions in which they post job openings. Sort the results by descending number.

In [53]:
# query text
query = f'''
    SELECT
        e.name,
        COUNT(DISTINCT v.area_id)
    FROM
        public.vacancies v LEFT JOIN
        public.employers e ON v.employer_id = e.id
    GROUP BY e.id
    ORDER BY COUNT(DISTINCT v.area_id) DESC
'''

In [54]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,name,count
0,Яндекс,181
1,Ростелеком,152
2,Спецремонт,116
3,Поляков Денис Иванович,88
4,ООО ЕФИН,71


4. Write a query to count the number of employers whose field of activity is not specified.

In [67]:
# query text
query = f'''
    SELECT *
    FROM
        public.employers e LEFT JOIN
        public.employers_industries ei ON e.id = ei.employer_id
    WHERE ei.industry_id IS NULL
'''

In [69]:
# query result
df = pd.read_sql_query(query, connection)
print('Number of employers whose field of activity is not specified:', len(df))

Number of employers whose field of activity is not specified: 8419


5. Write a query to find out the name of the company that is in third place in the alphabetical list (by name) of companies that have four areas of activity listed.

In [ ]:
# query text
query = f'''
    SELECT
        e.name,
        COUNT(ei.industry_id)
    FROM
        public.employers e LEFT JOIN
        public.employers_industries ei ON e.id = ei.employer_id
    GROUP BY e.id
    HAVING COUNT(ei.industry_id) = 4
    ORDER BY e.name
'''

In [71]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,name,count
0,101 Интернет,4
1,21vek.by,4
2,2ГИС,4
3,2К,4
4,4 пикселя +,4


6. Use this query to find out how many employers list Software Development as their field of activity.

In [83]:
# query text
query = f'''
    SELECT *
    FROM
        public.employers e JOIN
        public.employers_industries ei ON e.id = ei.employer_id JOIN
        public.industries i ON ei.industry_id = i.id
    WHERE i.name = 'Разработка программного обеспечения'
'''

In [84]:
# query result
df = pd.read_sql_query(query, connection)
print('Employers list "Software Development" as their field of activity:', len(df))

Employers list "Software Development" as their field of activity: 3553


7. For Yandex, display a list of regions with a population of over 1 million where the company has job openings, along with the number of openings in these regions. Also add a "Total" row with the total number of job openings for the company. Sort the results by ascending number.

The list of cities with a population of over 1 million should be taken from [here](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8).

If you have difficulty with this task, consult the PYTHON-17 module. How to retrieve data from web sources and APIs.

In [153]:
# code for getting a list of cities with a population of over a million
import requests
from bs4 import BeautifulSoup

url = 'https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
page = BeautifulSoup(response.text, 'html.parser')

# size of the required table
n = 16 * 6

# getting data from table
table_data = [city.text for city in page.find_all('td')][:n+1]

# getting city names from table (2nd column)
cities = []
for i, element in enumerate(table_data):
    if not (i-1) % 6:
        cities.append(element)
        print(element)

Москва
Санкт-Петербург
Новосибирск
Екатеринбург
Казань
Красноярск
Нижний Новгород
Челябинск
Уфа
Краснодар
Самара
Ростов-на-Дону
Омск
Воронеж
Пермь
Волгоград


In [154]:
# query text
query = f'''
    SELECT
        a.name,
        COUNT(v.id) AS cnt
    FROM
        public.vacancies v JOIN
        public.areas a ON v.area_id = a.id JOIN
        public.employers e ON v.employer_id = e.id
    WHERE
        a.name in {tuple(cities)} AND
        e.name = 'Яндекс'
    GROUP BY a.id
       
    UNION ALL SELECT
        'Total',
        COUNT(v.id)
    FROM
        public.vacancies v JOIN
        public.areas a ON v.area_id = a.id JOIN
        public.employers e ON v.employer_id = e.id
    WHERE
        a.name in {tuple(cities)} AND
        e.name = 'Яндекс'
    ORDER BY cnt
'''

In [155]:
# query result
df = pd.read_sql_query(query, connection)
df

,name,cnt
0,Омск,21
1,Челябинск,22
2,Красноярск,23
3,Волгоград,24
4,Пермь,25
5,Казань,25
6,Ростов-на-Дону,25
7,Самара,26
8,Уфа,26
9,Краснодар,30


***

In [4]:
# Conclusions from the analysis of employers

# Unit 6. Subject analysis

1. How many job openings are data-related?

We consider a job opening to be data-related if its title contains the words 'data' or 'данн'.

*Hint: Please note that job titles can be written in any case.*

In [11]:
# query text
query = f'''
    SELECT *
    FROM public.vacancies
    WHERE
        lower(name) LIKE '%data%' OR
        lower(name) LIKE '%данн%'
'''

In [13]:
# query result
df = pd.read_sql_query(query, connection)
print('Jobs are data related:', len(df))

Jobs are data related: 1771


2. How many suitable job openings are there for a junior data scientist?

We'll consider job openings for data scientists to be those with at least one of the following combinations in the title:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (do not include HTML-related openings here)
* 'machine learning'
* 'машинн%обучен%'

** In the following tasks, we'll continue working with openings with this condition.*

We'll consider the following openings to be junior-level specialists:
* the title contains the word 'junior' *or*
* required experience: No experience *or*
* type of employment: Internship.

In [42]:
# query text
query = f'''
    SELECT *
    FROM public.vacancies
    WHERE
        (lower(name) LIKE '%data scientist%' OR
        lower(name) LIKE '%data science%' OR
        lower(name) LIKE '%исследователь данных%' OR
        (lower(name) LIKE '%ML%' AND lower(name) NOT LIKE '%HTML%') OR
        lower(name) LIKE '%machine learning%' OR
        lower(name) LIKE '%машинн%обучен%')
        AND
        (lower(name) LIKE '%junior%' OR
        experience = 'Нет опыта' OR
        employment = 'Стажировка')
'''

In [43]:
# query result
df = pd.read_sql_query(query, connection)
print('Number of vacancies for Junior Data Scientist:', len(df))

Number of vacancies for Junior Data Scientist: 47


3. How many DS job openings are there that list SQL or Postgres as a key skill?

**The criteria for classifying a DS job opening are specified in the previous task.*

In [39]:
# query text
query = f'''
    SELECT *
    FROM public.vacancies
    WHERE
        (lower(name) LIKE '%data scientist%' OR
        lower(name) LIKE '%data science%' OR
        lower(name) LIKE '%исследователь данных%' OR
        (lower(name) LIKE '%ML%' AND lower(name) NOT LIKE '%HTML%') OR
        lower(name) LIKE '%machine learning%' OR
        lower(name) LIKE '%машинн%обучен%')
        AND
        (lower(key_skills) LIKE '%sql%' OR
        lower(key_skills) LIKE '%postgres%')
'''

In [40]:
# query result
df = pd.read_sql_query(query, connection)
print('Number of vacancies with key skill SQL or Postgres:', len(df))

Number of vacancies with key skill SQL or Postgres: 173


4. Check how popular Python is in employers' DS requirements. To do this, calculate the number of job postings that list Python as a key skill.

**This can be done using a query similar to the previous one.*

In [34]:
# query text
query = f'''
    SELECT *
    FROM public.vacancies
    WHERE
        (lower(name) LIKE '%data scientist%' OR
        lower(name) LIKE '%data science%' OR
        lower(name) LIKE '%исследователь данных%' OR
        (lower(name) LIKE '%ML%' AND lower(name) NOT LIKE '%HTML%') OR
        lower(name) LIKE '%machine learning%' OR
        lower(name) LIKE '%машинн%обучен%')
        AND
        lower(key_skills) LIKE '%python%'
'''

In [38]:
# query result
df = pd.read_sql_query(query, connection)
print('Number of vacancies with key skill Python:', len(df))

Number of vacancies with key skill Python: 292


5. How many key skills are listed on average in job postings for data scientists?
Round your answer to two decimal places.

In [75]:
# query text
query = f'''
    SELECT ROUND(AVG(char_length(key_skills) - char_length(replace(key_skills, CHR(9), '')) + 1), 2)
    FROM public.vacancies
    WHERE
        (lower(name) LIKE '%data scientist%' OR
        lower(name) LIKE '%data science%' OR
        lower(name) LIKE '%исследователь данных%' OR
        (lower(name) LIKE '%ML%' AND lower(name) NOT LIKE '%HTML%') OR
        lower(name) LIKE '%machine learning%' OR
        lower(name) LIKE '%машинн%обучен%')
        AND
        key_skills IS NOT NULL
'''

In [78]:
# query result
df = pd.read_sql_query(query, connection)
print('Key skills on average in DS job vacancies:', df['round'][0])

Key skills on average in DS job vacancies: 6.44


6. Write a query to calculate the **average** salary for DS for each type of required experience (a unique value from the *experience* field).

When solving this problem, consider the following:
1. We only consider jobs for which at least one of the two salary fields is filled in.
2. If both salary fields are filled in, we calculate the salary for each job as the sum of the two fields divided by 2. If only one field is filled in, we calculate that salary as the salary for the job.
3. If null is used in the calculations, the result will also be null (see what the query select 1 + null returns). To avoid this situation, we'll use the [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull) function, which will replace null with the value we pass. For example, see what the query `select 1 + coalesce(null, 0)` returns.

Find out the average salary for a data scientist with 3 to 6 years of experience. Round the result to the nearest whole number.

In [96]:
# query text
query = f'''
    SELECT
        experience,
        ROUND(AVG(coalesce(salary_from, salary_to) + coalesce(salary_to, salary_from)))
    FROM public.vacancies
    WHERE
        (lower(name) LIKE '%data scientist%' OR
        lower(name) LIKE '%data science%' OR
        lower(name) LIKE '%исследователь данных%' OR
        (lower(name) LIKE '%ML%' AND lower(name) NOT LIKE '%HTML%') OR
        lower(name) LIKE '%machine learning%' OR
        lower(name) LIKE '%машинн%обучен%')
        AND
        (salary_from IS NOT NULL OR
        salary_to IS NOT NULL)
    GROUP BY experience
'''

In [97]:
# query result
df = pd.read_sql_query(query, connection)
df.head()

,experience,round
0,От 3 до 6 лет,506258.0
1,Нет опыта,149286.0
2,От 1 года до 3 лет,293378.0


***

In [ ]:
# conclusions from the subject analysis

# General conclusion on the project

In [ ]:
# summarize the research, summarize the findings
# here you can (this will be a plus) conduct additional data analysis, make predictions, and consider options for continuing the research